In [ ]:
import warnings
import cartopy.crs as ccrs
import geoviews.feature as gf
import holoviews as hv
from holoviews import opts
import uxarray as ux
import numpy as np
hv.extension("bokeh")
# hv.extension("matplotlib")

maps = gf.coastline(projection=ccrs.PlateCarree(), line_width=1, scale="50m") \
         * gf.states(projection=ccrs.PlateCarree(), line_width=1, line_color='gray', scale="50m")

In [ ]:
grid_file="../data/samples/mpasjedi/invariant.nc"
bkg_file="../data/samples/mpasjedi/bkg.nc"
ana_file="../data/samples/mpasjedi/ana.nc"

uxds_a = ux.open_dataset(grid_file, ana_file)
uxds_b = ux.open_dataset(grid_file, bkg_file)
### uxdiff = uxds_a - uxds_b  # cannot diff the whole datasets, but we can diff each individual variable
uxds_a   # display information about the dataset

In [ ]:
def mpas_plot(ux1D, maps, title, minval=None, maxval=None):
    import math
    # Get min and max
    if minval is None:
        amin = ux1D.min().item()
        minval = math.floor(amin)
    if maxval is None:
        amax = ux1D.max().item()
        maxval = math.ceil(amax)
    title += f" min={amin:.1f} max={amax:.1f}"
        
    # generate contour plot (Notes: #line_width=0.001)
    #        ux1D.plot(), levels=np.arange(minval, maxval, 0.5), filled=True).opts(line_color=None) 
    contour_plot = hv.operation.contours(
        ux1D.plot(), levels=np.linspace(minval, maxval, num=20), filled=True
    ).opts(line_color=None) 
    
    final = contour_plot.opts(
        width=800, height=500, cmap='coolwarm', clim=(minval, maxval), colorbar=True,
        show_legend=False, tools=['hover'], title=title
    ) * maps
    return final

In [ ]:
%%time

uxvar = uxds_a['theta'] - 273.15

# generate color plot
# color_plot = ux1D.plot(cmap='inferno') * maps
nt = 0  # time dimension
lev = 0  # vertical level
plot = mpas_plot(uxvar.isel(Time=nt, nVertLevels=lev), maps, title=f'lev={lev}')
hv.save(plot, 'theta.png')
display(plot)

In [ ]:
var_name = "theta"
uxdiff = uxds_a[var_name] - uxds_b[var_name]  # calculate the differences

### subset
# lon_center = -105.03
# lat_center = 40.0
# lon_incr = 5 # degree
# lat_incr = 5 # degree
# lon_bounds = (lon_center - lon_incr, lon_center + lon_incr)
# lat_bounds = (cen_lat - lat_incr, cen_lat + lat_incr)
# uxdiffb = uxdiff.subset.bounding_box(lon_bounds, lat_bounds,)

uxvar = uxdiff  # uxdiffb
nt=0  # time dimension

plots = []
for lev in [0, 15, 30]:  # vertical levels
    tmp = mpas_plot(uxvar.isel(Time=nt, nVertLevels=lev), maps, title=f'lev={lev}')
    plots.append(tmp)
    
# hv.Layout(plots).cols(1)  # subplots share one toolbar, which facilitates doing sync'ed zoom-in/out

for p in plots:  # each subplot has its own toolbar
    display(p)